# Backfill author_registry.orcid from raw_orcid Conflicts

**Problem**: ~3.2M author entities have works carrying 2+ distinct `raw_orcid` values, indicating overmerged profiles. The current `author.orcid` in CreateAuthors uses "most recent work wins" which is unreliable.

**Approach**: 
1. For authors with a **clear majority** orcid (one orcid with strictly more works than any other), set `author_registry.orcid` to that orcid and NULL `author_id` on minority works.
2. For **ties**, attempt to break using **name matching** (compare `raw_author_name` to the profile's `display_name`). If one orcid's works match the profile name better, treat it as the majority and set registry orcid.
3. For **remaining ties**, NULL `author_id` on all orcid-tagged works (both sides) and let MatchAuthors reassign. No registry orcid is set.

**oxjobs**: #110 split-author-profiles-on-orcid

### Cell 1: Diagnostic — Count authors with conflicting ORCIDs

In [ ]:
WITH author_orcids AS (
    SELECT
        CAST(REPLACE(auth.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
        auth.raw_orcid,
        COUNT(*) AS work_count
    FROM openalex.works.openalex_works
    LATERAL VIEW POSEXPLODE(authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
    GROUP BY 1, 2
),
author_stats AS (
    SELECT
        author_id,
        COUNT(*) AS distinct_orcids,
        MAX(work_count) AS max_count,
        SUM(CASE WHEN work_count = max_wc THEN 1 ELSE 0 END) AS num_at_max
    FROM (
        SELECT *, MAX(work_count) OVER (PARTITION BY author_id) AS max_wc
        FROM author_orcids
    )
    GROUP BY author_id
    HAVING COUNT(*) >= 2
)
SELECT
    COUNT(*) AS authors_with_multiple_orcids,
    SUM(CASE WHEN num_at_max = 1 THEN 1 ELSE 0 END) AS clear_majority,
    SUM(CASE WHEN num_at_max > 1 THEN 1 ELSE 0 END) AS ties
FROM author_stats

### Cell 2: Build majority orcid lookup (clear majority only)

For each author with 2+ distinct raw_orcids where one orcid has strictly more works than any other.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.orcid_majority_lookup AS
WITH author_orcids AS (
    SELECT
        CAST(REPLACE(auth.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
        auth.raw_orcid,
        COUNT(*) AS work_count
    FROM openalex.works.openalex_works
    LATERAL VIEW POSEXPLODE(authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
    GROUP BY 1, 2
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY work_count DESC) AS rn,
        COUNT(*) OVER (PARTITION BY author_id) AS distinct_orcids
    FROM author_orcids
),
with_second AS (
    SELECT
        r1.author_id,
        r1.raw_orcid AS majority_orcid,
        r1.work_count AS majority_count,
        r2.work_count AS second_count,
        r1.distinct_orcids
    FROM ranked r1
    JOIN ranked r2 ON r1.author_id = r2.author_id AND r2.rn = 2
    WHERE r1.rn = 1
      AND r1.distinct_orcids >= 2
)
SELECT
    author_id,
    majority_orcid,
    majority_count,
    second_count,
    distinct_orcids
FROM with_second
WHERE majority_count > second_count

### Cell 3: Inspect lookup table

In [ ]:
SELECT
    COUNT(*) AS total_authors,
    SUM(second_count) AS total_minority_works
FROM openalex.authors.orcid_majority_lookup

In [ ]:
SELECT ol.author_id, ol.majority_orcid, ol.majority_count, ol.second_count, ol.distinct_orcids,
       ar.display_name, ar.orcid AS current_registry_orcid
FROM openalex.authors.orcid_majority_lookup ol
LEFT JOIN openalex.authors.author_registry ar ON ol.author_id = ar.id
ORDER BY ol.second_count DESC
LIMIT 30

### Cell 4: Name matching tiebreaker for ties

For authors where orcid work counts are tied, compare `raw_author_name` on each orcid's works to the profile's `display_name`. If one orcid has more exact name matches, treat it as the majority and add it to the lookup table.

In [ ]:
WITH author_orcids AS (
    SELECT
        CAST(REPLACE(auth.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
        auth.raw_orcid,
        COUNT(*) AS work_count
    FROM openalex.works.openalex_works
    LATERAL VIEW POSEXPLODE(authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
    GROUP BY 1, 2
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY work_count DESC) AS rn
    FROM author_orcids
),
tied_authors AS (
    SELECT r1.author_id, r1.work_count AS tied_count
    FROM ranked r1
    JOIN ranked r2 ON r1.author_id = r2.author_id AND r2.rn = 2
    WHERE r1.rn = 1
      AND r1.work_count = r2.work_count
      AND r1.author_id NOT IN (SELECT author_id FROM openalex.authors.orcid_majority_lookup)
),
exploded AS (
    SELECT
        CAST(REPLACE(auth.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
        auth.raw_orcid,
        auth.raw_author_name
    FROM openalex.works.openalex_works
    LATERAL VIEW POSEXPLODE(authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
),
name_scores AS (
    SELECT
        e.author_id,
        e.raw_orcid,
        SUM(CASE WHEN LOWER(TRIM(e.raw_author_name)) = LOWER(TRIM(oa.display_name)) THEN 1 ELSE 0 END) AS exact_matches,
        COUNT(*) AS works
    FROM exploded e
    INNER JOIN tied_authors ta ON e.author_id = ta.author_id
    INNER JOIN openalex.authors.openalex_authors oa ON ta.author_id = oa.id
    GROUP BY 1, 2
),
name_ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY exact_matches DESC) AS rn,
        COUNT(*) OVER (PARTITION BY author_id) AS distinct_orcids
    FROM name_scores
),
name_winners AS (
    SELECT
        r1.author_id,
        r1.raw_orcid AS majority_orcid,
        r1.exact_matches AS best_matches,
        r2.exact_matches AS second_matches,
        r1.works AS majority_count,
        r2.works AS second_count,
        r1.distinct_orcids
    FROM name_ranked r1
    JOIN name_ranked r2 ON r1.author_id = r2.author_id AND r2.rn = 2
    WHERE r1.rn = 1
      AND r1.exact_matches > r2.exact_matches
)
INSERT INTO openalex.authors.orcid_majority_lookup
SELECT author_id, majority_orcid, majority_count, second_count, distinct_orcids
FROM name_winners

In [ ]:
SELECT
    COUNT(*) AS total_authors_in_lookup,
    'after adding name-match tiebreakers' AS note
FROM openalex.authors.orcid_majority_lookup

### Cell 5: Backfill author_registry.orcid

Sets `orcid` on author_registry to the majority raw_orcid for all authors in the lookup (clear majority + name-match tiebreakers).

In [ ]:
MERGE INTO openalex.authors.author_registry AS target
USING openalex.authors.orcid_majority_lookup AS source
ON target.id = source.author_id
WHEN MATCHED THEN UPDATE SET
    target.orcid = source.majority_orcid,
    target.updated_date = current_timestamp()

### Cell 5b: Verification

In [ ]:
-- Confirm registry orcids were set
SELECT COUNT(*) AS authors_with_registry_orcid
FROM openalex.authors.author_registry
WHERE orcid IS NOT NULL

In [ ]:
-- Spot-check: compare registry orcid to current openalex_authors orcid
SELECT ar.id AS author_id, ar.display_name, ar.orcid AS registry_orcid,
       oa.orcid AS current_profile_orcid,
       CASE WHEN ar.orcid = oa.orcid THEN 'match' ELSE 'different' END AS comparison
FROM openalex.authors.author_registry ar
JOIN openalex.authors.openalex_authors oa ON ar.id = oa.id
WHERE ar.orcid IS NOT NULL
ORDER BY RAND(42)
LIMIT 30

### Cell 6: Build works repair table

Two sources:
1. **Majority authors** — works where `raw_orcid` doesn't match the majority orcid (minority works only)
2. **Tied authors** — ALL orcid-tagged works (both sides get NULLed since we can't pick a winner)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.orcid_minority_works_repair AS
WITH author_orcids AS (
    SELECT
        CAST(REPLACE(auth.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
        auth.raw_orcid,
        COUNT(*) AS work_count
    FROM openalex.works.openalex_works
    LATERAL VIEW POSEXPLODE(authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
    GROUP BY 1, 2
),
tied_authors AS (
    SELECT author_id
    FROM (
        SELECT *,
            MAX(work_count) OVER (PARTITION BY author_id) AS max_wc
        FROM author_orcids
    )
    GROUP BY author_id
    HAVING COUNT(*) >= 2
       AND SUM(CASE WHEN work_count = max_wc THEN 1 ELSE 0 END) > 1
       AND author_id NOT IN (SELECT author_id FROM openalex.authors.orcid_majority_lookup)
),
exploded AS (
    SELECT
        w.id AS work_id,
        pos AS author_sequence,
        CAST(REPLACE(auth.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
        auth.raw_orcid
    FROM openalex.works.openalex_works w
    LATERAL VIEW POSEXPLODE(w.authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
)
-- Part 1: minority works for authors with a clear majority
SELECT
    e.work_id,
    e.author_sequence,
    e.author_id,
    e.raw_orcid,
    m.majority_orcid
FROM exploded e
INNER JOIN openalex.authors.orcid_majority_lookup m ON e.author_id = m.author_id
WHERE e.raw_orcid != m.majority_orcid
UNION ALL
-- Part 2: all orcid-tagged works for tied authors
SELECT
    e.work_id,
    e.author_sequence,
    e.author_id,
    e.raw_orcid,
    NULL AS majority_orcid
FROM exploded e
INNER JOIN tied_authors ta ON e.author_id = ta.author_id

### Cell 7: Inspect minority works repair table

In [ ]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT work_id) AS distinct_works,
    COUNT(DISTINCT author_id) AS distinct_authors,
    SUM(CASE WHEN majority_orcid IS NOT NULL THEN 1 ELSE 0 END) AS majority_minority_works,
    SUM(CASE WHEN majority_orcid IS NULL THEN 1 ELSE 0 END) AS tied_author_works
FROM openalex.authors.orcid_minority_works_repair

In [ ]:
SELECT r.work_id, r.author_sequence, r.author_id, r.raw_orcid, r.majority_orcid,
       wa.raw_author_name,
       oa.display_name AS profile_display_name
FROM openalex.authors.orcid_minority_works_repair r
LEFT JOIN openalex.works.work_authors wa
    ON r.work_id = wa.work_id AND r.author_sequence = wa.author_sequence
LEFT JOIN openalex.authors.openalex_authors oa
    ON r.author_id = oa.id
ORDER BY RAND(42)
LIMIT 30

### Cell 8: NULL author_id on minority works

Safety guard: joins on `author_id` so we only NULL if the assignment hasn't changed since we built the repair table.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.orcid_minority_works_repair AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
   AND target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 9: Queue affected works for reprocessing

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync (work_id, curated_ras, added_datetime)
SELECT DISTINCT
    work_id,
    'orcid_split_repair' AS curated_ras,
    current_timestamp() AS added_datetime
FROM openalex.authors.orcid_minority_works_repair
WHERE NOT EXISTS (
    SELECT 1 FROM openalex.institutions.curated_work_ids_pending_sync p
    WHERE p.work_id = openalex.authors.orcid_minority_works_repair.work_id
)

### Cell 10: Final verification

In [ ]:
-- Confirm affected rows are now NULL
SELECT COUNT(*) AS nulled_rows
FROM openalex.works.work_authors wa
JOIN openalex.authors.orcid_minority_works_repair r
    ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
WHERE wa.author_id IS NULL

In [ ]:
-- Confirm sync queue populated
SELECT COUNT(DISTINCT work_id) AS queued_works
FROM openalex.institutions.curated_work_ids_pending_sync
WHERE curated_ras = 'orcid_split_repair'